In [13]:
# STEP 1: Import libraries
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [15]:
# STEP 2: Set parameters
VOCAB_SIZE = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 32
HIDDEN_SIZE = 32
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 0.001
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print("Using device:", device)

In [17]:

# STEP 3: Load the IMDb dataset
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("First review as numbers:", X_train[0][:20])
print("First review label:", y_train[0])

# STEP 4: Pad or truncate reviews to equal length
X_train = pad_sequences(X_train, maxlen=MAX_LENGTH, padding="post", truncating="post")
X_test = pad_sequences(X_test, maxlen=MAX_LENGTH, padding="post", truncating="post")

# STEP 5: Convert NumPy arrays to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# STEP 6: Create DataLoaders for mini-batch training
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# STEP 7: Define the RNN model
class SentimentRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM, padding_idx=0)
        self.rnn = nn.RNN(EMBEDDING_DIM, HIDDEN_SIZE, nonlinearity="tanh", batch_first=True)
        self.fc = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        lengths = (x != 0).sum(dim=1)  # Count actual tokens in each review
        x = self.embedding(x)          # Convert word IDs to vectors

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, hidden = self.rnn(packed)    # Ignore padding positions
        logits = self.fc(hidden[-1])    # Use the final valid hidden state
        return logits.squeeze(1)


# STEP 8: Create model, loss function and optimizer
model = SentimentRNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

# STEP 9: Train the RNN
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for reviews, labels in train_loader:
        reviews, labels = reviews.to(device), labels.to(device)

        optimizer.zero_grad()             # Clear old gradients
        outputs = model(reviews)          # Forward pass
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                   # Backpropagation through time
        optimizer.step()                  # Update weights

        total_loss += loss.item() * labels.size(0)
        predictions = (outputs >= 0).float()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}/{EPOCHS}, "
          f"Loss: {total_loss/total:.4f}, "
          f"Accuracy: {correct/total:.4f}")

# STEP 10: Evaluate the model on test data
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for reviews, labels in test_loader:
        reviews, labels = reviews.to(device), labels.to(device)
        outputs = model(reviews)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * labels.size(0)
        predictions = (outputs >= 0).float()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print("\nTest loss:", round(test_loss/total, 4))
print("Test accuracy:", round(correct/total, 4))

# STEP 11: Encode new reviews using IMDb's word index
word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    sequence = [1]  # Start-of-review token

    for word in words:
        if word in word_index:
            index = word_index[word] + 3
            if index >= VOCAB_SIZE:
                index = 2
        else:
            index = 2  # Correct unknown-word token

        sequence.append(index)

    sequence = pad_sequences(
        [sequence], maxlen=MAX_LENGTH,
        padding="post", truncating="post"
    )

    return torch.tensor(sequence, dtype=torch.long)


# STEP 12: Predict sentiment for new reviews
def predict_sentiment(text):
    model.eval()
    review_tensor = encode_review(text).to(device)

    with torch.no_grad():
        logits = model(review_tensor)
        probability = torch.sigmoid(logits).item()

    sentiment = "Positive" if probability >= 0.5 else "Negative"
    print("\nReview:", text)
    print("Positive probability:", round(probability, 4))
    print("Predicted sentiment:", sentiment)

predict_sentiment("This movie was excellent and wonderful")
predict_sentiment("This movie was terrible and boring")


Training samples: 25000
Testing samples: 25000
First review as numbers: [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
First review label: 1
SentimentRNN(
  (embedding): Embedding(10000, 32, padding_idx=0)
  (rnn): RNN(32, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
Epoch 1/5, Loss: 0.6898, Accuracy: 0.5347
Epoch 2/5, Loss: 0.6594, Accuracy: 0.6056
Epoch 3/5, Loss: 0.6369, Accuracy: 0.6387
Epoch 4/5, Loss: 0.6096, Accuracy: 0.6771
Epoch 5/5, Loss: 0.5752, Accuracy: 0.7107

Test loss: 0.617
Test accuracy: 0.6897

Review: This movie was excellent and wonderful
Positive probability: 0.6061
Predicted sentiment: Positive

Review: This movie was terrible and boring
Positive probability: 0.1264
Predicted sentiment: Negative
